# Installing Libaries

In [ ]:
!pip install openpyxl

In [ ]:
pip install tabulate


# Importing Libaries

In [ ]:
import pandas as pd
import random
from tabulate import tabulate
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
def read_courses(file_path):
    return pd.read_excel(file_path)

def read_doctor_availability(file_path):
    return pd.read_excel(file_path)

def generate_master_timetable(courses_df, availability_df):
    # Define timeslots
    timeslots = [
        "Saturday_9AM", "Saturday_10:40AM", "Saturday_1:10PM", "Saturday_2:50PM",
        "Sunday_9AM", "Sunday_10:40AM", "Sunday_1:10PM", "Sunday_2:50PM",
        "Monday_9AM", "Monday_10:40AM", "Monday_1:10PM", "Monday_2:50PM",
        "Tuesday_9AM", "Tuesday_10:40AM", "Tuesday_1:10PM", "Tuesday_2:50PM",
        "Wednesday_9AM", "Wednesday_10:40AM", "Wednesday_1:10PM", "Wednesday_2:50PM",
        "Thursday_9AM", "Thursday_10:40AM", "Thursday_1:10PM", "Thursday_2:50PM"
    ]
    
def export_master_timetable(master_timetable, file_path):
    # Create a DataFrame from the master timetable dictionary
    timetable_data = []
    for timeslot, courses in master_timetable.items():
        for course in courses:
            timetable_data.append({'Timeslot': timeslot, 'Course ID': course[0], 'Course Name': course[1], 'Doctor Name': course[2]})
    timetable_df = pd.DataFrame(timetable_data)
    
    # Export the DataFrame to an Excel file
    timetable_df.to_excel(file_path, index=False)

# Data Visualization

Count Plot of Courses by Doctor

In [ ]:
def visualize_courses_by_doctor(courses_df):
    plt.figure(figsize=(12, 6))
    sns.countplot(data=courses_df, x='Doctors_Names', order=courses_df['Doctors_Names'].value_counts().index)
    plt.title("Number of Courses by Doctor")
    plt.xlabel("Doctor")
    plt.ylabel("Number of Courses")
    plt.xticks(rotation=45, ha='right')
    plt.show()

# Example usage
if __name__ == "__main__":
    courses_df = read_courses("Courses.xlsx")
    visualize_courses_by_doctor(courses_df)

Pie Chart of Course Distribution by Department

In [ ]:
def visualize_course_distribution_by_department(courses_df):
    plt.figure(figsize=(20, 12))
    courses_df['Department'] = courses_df['course_id'].str[:3]  # Assuming course_id starts with department code
    department_counts = courses_df['Department'].value_counts()
    department_counts.plot.pie(autopct='%1.1f%%', startangle=140, cmap='viridis')
    plt.title("Course Distribution by Department")
    plt.ylabel("")  # Hide y-label
    plt.show()

# Example usage
if __name__ == "__main__":
    courses_df = read_courses("Courses.xlsx")
    visualize_course_distribution_by_department(courses_df)


# Result

In [ ]:
def read_courses(file_path):
    return pd.read_excel(file_path)

def read_doctor_availability(file_path):
    return pd.read_excel(file_path)

def generate_master_timetable(courses_df, availability_df):
    # Define timeslots
    timeslots = [
        "Saturday_9AM", "Saturday_10:40AM", "Saturday_1:10PM", "Saturday_2:50PM",
        "Sunday_9AM", "Sunday_10:40AM", "Sunday_1:10PM", "Sunday_2:50PM",
        "Monday_9AM", "Monday_10:40AM", "Monday_1:10PM", "Monday_2:50PM",
        "Tuesday_9AM", "Tuesday_10:40AM", "Tuesday_1:10PM", "Tuesday_2:50PM",
        "Wednesday_9AM", "Wednesday_10:40AM", "Wednesday_1:10PM", "Wednesday_2:50PM",
        "Thursday_9AM", "Thursday_10:40AM", "Thursday_1:10PM", "Thursday_2:50PM"
    ]
    
    # Shuffle timeslots to randomize assignment
    random.shuffle(timeslots)
    
    # Create a master timetable dictionary
    master_timetable = {timeslot: [] for timeslot in timeslots}
    
    # Assign courses to timeslots while considering constraints
    for index, row in courses_df.iterrows():
        course_id = row['course_id'].strip()
        course_name = row['course_name']
        doctor_name = row['Doctors_Names']
        available_days = availability_df[availability_df['Doctors_Names'] == doctor_name]['available'].tolist()
        available_timeslots = [day + '_' + timeslot.split('_')[1] for day in available_days for timeslot in timeslots if day in timeslot]
        for timeslot in available_timeslots:
            if len(master_timetable[timeslot]) < 2:  # Check if the timeslot has less than 2 courses
                master_timetable[timeslot].append((course_id, course_name, doctor_name))
                break
    
    return master_timetable

def export_master_timetable(master_timetable, file_path):
    # Create a DataFrame from the master timetable dictionary
    timetable_data = []
    for timeslot, courses in master_timetable.items():
        for course in courses:
            timetable_data.append({'Timeslot': timeslot, 'Course ID': course[0], 'Course Name': course[1], 'Doctor Name': course[2]})
    timetable_df = pd.DataFrame(timetable_data)
    
    # Export the DataFrame to an Excel file
    timetable_df.to_excel(file_path, index=False)

def get_registered_courses(student_id, file_path):
    df = pd.read_excel(file_path)
    # Clean up the data by removing leading/trailing whitespace and "\t" characters
    df.columns = df.columns.str.strip().str.replace('\t', '')
    df['course_id'] = df['course_id'].str.strip().str.replace('\t', '')
    df['course_name'] = df['course_name'].str.strip().str.replace('\t', '')
    df['Student_id'] = df['Student_id'].astype(str).str.strip().str.replace('\t', '')
    registered_courses = df[df['Student_id'] == student_id]
    
    if registered_courses.empty:
        print("No registered courses found for the provided student ID.")
        return None, None  # Return None for both registered_courses and classroom_names
    # Read classroom names
    classroom_names = df[['course_id', 'class room_name']].drop_duplicates()
    return registered_courses, classroom_names


def retrieve_assigned_times(reg_courses, master_timetable):
    assigned_times = []
    for index, row in reg_courses.iterrows():
        course_id = row['course_id']
        for timeslot, courses in master_timetable.items():
            for course in courses:
                if course[0] == course_id:
                    assigned_times.append((course_id, course[1], course[2], timeslot))
    return assigned_times

def resolve_conflicts(assigned_times, master_timetable):
    timeslots_used = set()
    for course in assigned_times:
        timeslot = course[3]
        if timeslot in timeslots_used:
            for alt_timeslot in master_timetable.keys():
                if alt_timeslot not in timeslots_used:
                    timeslots_used.add(alt_timeslot)
                    assigned_times[assigned_times.index(course)] = (course[0], course[1], course[2], alt_timeslot)
                    break
        else:
            timeslots_used.add(timeslot)
    return assigned_times

if __name__ == "__main__":
    # Read data
    courses_df = read_courses("Courses.xlsx")
    availability_df = read_doctor_availability("Doctors_availability.xlsx")
    
    # Generate master timetable
    master_timetable = generate_master_timetable(courses_df, availability_df)
    
    # Export master timetable
    export_master_timetable(master_timetable, "Master_Timetable.xlsx")

In [ ]:
student_id = input("Enter student ID: ").strip()

# Retrieve registered courses
registered_courses, classroom_names = get_registered_courses(student_id, "IDs&Courses.xlsx")

if registered_courses is not None:
    # Display registered courses in a table format
    print("Registered Courses:")
    print(tabulate(registered_courses, headers='keys', tablefmt='pretty'))

    # Retrieve assigned times for registered courses
    assigned_times = retrieve_assigned_times(registered_courses, master_timetable)

    # Resolve conflicts
    resolved_times = resolve_conflicts(assigned_times, master_timetable)

    # Print assigned times
    print("\nAssigned Times:")
    for course in resolved_times:
        # Find classroom name
        classroom_name = classroom_names[classroom_names['course_id'] == course[0]]['class room_name'].iloc[0]
        print(f"Course ID: {course[0]:<10} Course Name: {course[1]:<50} Doctor: {course[2]:<20} Timeslot: {course[3]:<20} Classroom: {classroom_name}")
else:
    print("No registered courses found for the entered student ID.")


In [ ]:
def convert_timeslot(timeslot):
    day, time_str = timeslot.split('_')
    parts = time_str[:-2].split(':')
    hour = int(parts[0])
    minute = int(parts[1]) if len(parts) > 1 else 0  # Default to 0 minutes if not present
    if time_str.endswith("PM") and hour != 12:
        hour += 12
    return day, hour, minute


def print_courses_for_day(day, master_timetable):
    print(f"\nSchedule for {day}:")
    data = []
    for timeslot, courses in master_timetable.items():
        if day in timeslot:
            for course_id, course_name, doctor_name in courses:  # Unpack all 3 elements
                classroom_location = classroom_locations.get(course_id, "Location Unknown")
                data.append([course_id, course_name, doctor_name, classroom_location, timeslot])

    data.sort(key=lambda row: convert_timeslot(row[4]))
    print(tabulate(data, headers=["Course ID", "Course Name", "Doctor", "Classroom", "Timeslot"], tablefmt="grid"))



if __name__ == "__main__":
    courses_df = read_courses("Courses.xlsx")
    availability_df = read_doctor_availability("Doctors_availability.xlsx")
    master_timetable = generate_master_timetable(courses_df, availability_df)
    
    full_course_data = pd.read_excel("IDs&Courses.xlsx")
    full_course_data.rename(columns={'course_id\t': 'course_id'}, inplace=True)

    full_course_data['course_id'] = full_course_data['course_id'].str.strip()
    full_course_data['class room_name'] = full_course_data['class room_name'].str.strip()

    id_column = full_course_data.columns[0] 
    classroom_locations = dict(zip(full_course_data['course_id'], full_course_data['class room_name']))

    for day in ["Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday"]:
        print_courses_for_day(day, master_timetable)